# DOH Vision — 골프 클럽/공 검출 학습 (Colab)

`club_detect.html`에서 쓸 **best.onnx** 를 만드는 노트북.
**셀을 위에서부터 순서대로 실행**하면 됩니다. (런타임 → GPU 필수)

근거: `DOH_Vision_Club_Detection_Plan_v1.0.md`


## 0) GPU 확인
메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU** 로 설정 후 실행.


In [ ]:
!nvidia-smi


## 1) 라이브러리 설치


In [ ]:
!pip -q install ultralytics roboflow onnx onnxslim


## 2) 데이터셋 준비

**방법 A — Roboflow 자동 다운로드** (무료 API 키: roboflow.com → Settings → API Key)

> ⚠️ **라이선스 확인**: 각 데이터셋 페이지에서 상업적 사용 가능 여부를 반드시 확인하세요.
> 상업 제약이 있으면 자체 라벨 데이터로 대체/보강해야 합니다.

아래 workspace/project/version 은 예시입니다. 데이터셋 페이지의 실제 슬러그로 바꾸세요.


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")   # ← 본인 키 입력
# 예시: golf-club-tracking (6,750장). 실제 슬러그/버전으로 교체하세요.
project = rf.workspace("club-head-tracking").project("golf-club-tracking")
dataset = project.version(2).download("yolov11")
DATA = dataset.location + "/data.yaml"
print("data.yaml =", DATA)


**방법 B — 직접 업로드한 데이터셋 사용** (Roboflow 안 쓸 때)
왼쪽 파일창에 data.yaml/이미지/라벨을 올리고 아래 경로만 지정.


In [ ]:
# DATA = "/content/mydata/data.yaml"   # 방법 B 쓸 때 주석 해제


## 3) 클래스 순서 확인 (중요)
`club_detect.html` 의 **클래스 입력과 순서가 반드시 일치**해야 합니다. (기본 `clubhead,ball`)


In [ ]:
import yaml
with open(DATA) as f: cfg = yaml.safe_load(f)
print("names =", cfg.get("names"))
print("→ 이 순서를 club_detect.html '클래스' 입력에 그대로 넣으세요.")


## 4) 학습 (YOLO11n)
GPU에서 보통 수십 분~수시간. epochs는 데이터 규모에 맞게 조정.


In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(
    data=DATA, epochs=100, imgsz=640, batch=16,
    name="doh_club", patience=20, close_mosaic=10,
    # 임팩트 모션블러 대응 증강
    degrees=5.0, translate=0.1, scale=0.5, hsv_v=0.5,
)


## 5) ONNX export (브라우저 onnxruntime-web 용)
`opset=12, simplify=True, dynamic=False, nms=False` (NMS는 브라우저 JS가 수행).


In [ ]:
onnx_path = model.export(format="onnx", opset=12, imgsz=640, simplify=True, dynamic=False, nms=False)
print("ONNX =", onnx_path)


## 6) best.onnx 내려받기 → 호스팅
내려받은 파일을 **CORS 허용 호스팅**(예: 자체 서버, Cloudflare R2, 또는 GitHub raw)에 올리고,
그 URL을 `club_detect.html` 의 **모델 URL**에 입력하면 브라우저에서 자동 검출됩니다.


In [ ]:
from google.colab import files
files.download(onnx_path)


## 7) 검증
1. `club_detect.html` 열기 → 모델 URL + 클래스(3번의 names 순서) 입력 → **모델 로드**
2. **측면(DTL)** 스윙 영상 선택 → 분석
3. clubhead 빨강 박스 + 노란 샤프트선이 뜨는지, 검출률/샤프트각 확인
4. `analyzer2.html` **직접 그리기** 수동값과 대조해 오차 확인

OK면 다음 단계: analyzer2 의 Hough 샤프트를 이 YOLO 검출로 교체(v3) → 클럽패스·헤드스피드·P2/P6/P8 확장.
